# 🚨 Smart Emergency Dispatch — SFT → GRPO Training Pipeline

This notebook fine-tunes **Qwen3-1.7B** to act as an emergency 911 dispatcher:
1. **Phase 1 — SFT**: Teach the model the output format using rule-based examples
2. **Phase 2 — GRPO**: Improve strategy via reinforcement learning with the live environment

Built on [OpenEnv](https://github.com/meta-pytorch/OpenEnv) + [TRL](https://github.com/huggingface/trl).

### Setup — HuggingFace Spaces with GPU
1. Create a new Space → **Docker** SDK → **JupyterLab** template
2. Go to **Settings → Hardware** → select **A10G** or **A100**
3. Enable **Persistent Storage** (saves checkpoints to `/data`)
4. Add your `HF_TOKEN` in **Settings → Variables and secrets**
5. Open JupyterLab, upload this notebook, and run it

## 0 · Install Dependencies

In [ ]:
import subprocess, sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-Uq",
    "git+https://github.com/huggingface/trl.git",
    "git+https://github.com/meta-pytorch/OpenEnv.git",
    "trackio", "vllm==0.10.2", "bitsandbytes", "peft", "datasets",
    "git+https://github.com/rishiraj38/Smart_Emergency.git",
])

In [ ]:
import os
from huggingface_hub import login

# On HF Spaces the token is injected automatically as HF_TOKEN.
# Locally or on Colab, set it manually or use notebook_login().
hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(token=hf_token)
    print("✅ Logged in via HF_TOKEN")
else:
    from huggingface_hub import notebook_login
    notebook_login()

## 1 · Configuration

In [ ]:
import os, json, re, random
from collections import defaultdict

MODEL_NAME = "Qwen/Qwen3-1.7B"

# Use /data on HF Spaces (persistent storage) — falls back to local dirs elsewhere
_PERSIST = "/data" if os.path.isdir("/data") else "."
SFT_OUTPUT_DIR = os.path.join(_PERSIST, "smart-emergency-sft")
GRPO_OUTPUT_DIR = os.path.join(_PERSIST, "smart-emergency-grpo")

ENV_URL = "https://rishi38-eme-enviro.hf.space"  # ← update if self-hosting
print(f"Checkpoints will be saved to: {_PERSIST}/")

## 2 · System Prompt

In [ ]:
SYSTEM_PROMPT = """\
You are an expert 911 emergency dispatcher. You receive incoming calls and must make rapid, structured dispatch decisions.

## RULES
1. Each step you see: an incoming call transcript, active events, unit status, and a city map.
2. You must respond with a single JSON object — nothing else.

## ACTION TYPES
You have three action types: `dispatch`, `duplicate`, and `hold`.

### 1. dispatch — Handle a new emergency
Use when a FREE vehicle of the correct type is available.
```json
{
  "action_type": "dispatch",
  "severity_pred": <int 1-5>,
  "is_duplicate": false,
  "duplicate_of_event_id": null,
  "vehicle_type": "police" | "ambulance" | "fire",
  "vehicle_id": "<unit_id of a FREE vehicle>",
  "reroute": null
}
```

### 2. duplicate — Flag a repeat call
Use when the incoming call matches an existing active event (same location/type).
```json
{
  "action_type": "duplicate",
  "severity_pred": <int 1-5>,
  "is_duplicate": true,
  "duplicate_of_event_id": "<EVT-NNNN>",
  "vehicle_type": null,
  "vehicle_id": null,
  "reroute": null
}
```

### 3. hold — Queue for a busy vehicle
Use ONLY when ALL vehicles of the required type are busy (none are FREE). The event is queued and the named vehicle will auto-dispatch once it finishes its current job.
```json
{
  "action_type": "hold",
  "severity_pred": <int 1-5>,
  "is_duplicate": false,
  "duplicate_of_event_id": null,
  "vehicle_type": "police" | "ambulance" | "fire",
  "vehicle_id": "<unit_id of a BUSY vehicle to queue behind>",
  "reroute": null
}
```
**Hold rules:**
- NEVER hold if a free unit of the correct type exists — that gives a heavy penalty.
- Pick the vehicle that will free up soonest (lowest ETA) for a bonus.
- If the new call has HIGHER severity than what the busy units are handling, consider a reroute instead.

## REROUTE (optional, only with dispatch)
When dispatching, you may optionally reroute an in-flight vehicle from a LOWER-severity event to this HIGHER-severity one. Add a `reroute` block:
```json
{
  "action_type": "dispatch",
  "severity_pred": 4,
  "is_duplicate": false,
  "duplicate_of_event_id": null,
  "vehicle_type": "ambulance",
  "vehicle_id": "<any FREE unit for this call>",
  "reroute": {
    "vehicle_to_reroute": "<unit_id of a DISPATCHED vehicle>",
    "from_event_id": "<EVT-NNNN it is currently heading to>",
    "replacement_vehicle_id": "<FREE unit to cover the abandoned event, or null>"
  }
}
```
**Reroute rules:**
- Only reroute a vehicle with status DISPATCHED (not ON_SCENE or RETURNING).
- Only reroute FROM a lower-severity event TO a higher-severity one (+2 severity difference = best reward).
- Providing a valid replacement vehicle for the abandoned event earns a bonus.
- Invalid reroutes (wrong vehicle, wrong event ID) are penalised.

## SEVERITY GUIDE
1 = minor (shoplifting, twisted ankle)
2 = moderate (fainting, small dumpster fire, fender bender)
3 = serious (chest pains, house fire, mugging with weapon, cyclist hit)
4 = critical (unconscious person, trapped in wreck, gunshots, building fire with people)
5 = catastrophic (mass casualty, active shooter, multi-car pileup, city-block fire)

## VEHICLE GUIDE
- **fire** → fire, smoke, flames, gas leak
- **police** → shooting, robbery, fight, break-in, shoplifting
- **ambulance** → medical, crash, accident, injury, collapse

## STRATEGY
- Always pick the FREE vehicle closest to the incident (use CITY REFERENCE distances).
- If the call's location and type match an ACTIVE EVENT, it is likely a duplicate.
- Never dispatch a BUSY vehicle — use `hold` if no free units are available.
- Use `reroute` when a new call is much more severe than what a dispatched vehicle is handling.
- When holding, pick the vehicle with the lowest ETA to minimise wait time.
"""

## 3 · Connect to Environment

In [ ]:
from smart_emergency import SmartEmergencyEnv, SmartEmergencyAction

# EnvClient is async by default — .sync() gives us a synchronous wrapper
env = SmartEmergencyEnv(base_url=ENV_URL).sync()
# Quick smoke test
_test = env.reset()
print(f"✅ Connected — first call: {_test.observation.call_id}")

---
# Phase 1 — Supervised Fine-Tuning (SFT)
We run a heuristic agent, capture ground truth from the environment, and train the model to output correct JSON actions.

### 3.1 · Observation Parsing Helpers

In [ ]:
def parse_free_vehicles(obs_text: str) -> dict:
    """Return {unit_id: vehicle_type} for FREE vehicles."""
    vehicles = {}
    in_section = False
    for line in obs_text.split("\n"):
        if "=== UNIT STATUS ===" in line:
            in_section = True
            continue
        if in_section and line.startswith("==="):
            break
        if in_section and "|" in line and "FREE" in line:
            parts = [p.strip() for p in line.split("|")]
            if len(parts) >= 2:
                vehicles[parts[0]] = parts[1]
    return vehicles


def parse_all_vehicles(obs_text: str) -> list[dict]:
    """Return list of {id, type, status, event} for ALL vehicles."""
    vehicles = []
    in_section = False
    for line in obs_text.split("\n"):
        if "=== UNIT STATUS ===" in line:
            in_section = True
            continue
        if in_section and line.startswith("==="):
            break
        if in_section and "|" in line:
            parts = [p.strip() for p in line.split("|")]
            if len(parts) >= 4:
                status_str = parts[3]
                status = status_str.split("\u2192")[0].strip().split()[0] if status_str else "UNKNOWN"
                event = None
                if "\u2192" in status_str:
                    event = status_str.split("\u2192")[-1].strip()
                vehicles.append({"id": parts[0], "type": parts[1], "status": status, "event": event})
    return vehicles


def parse_active_events(obs_text: str) -> dict:
    """Return {EVT-NNNN: event_type}."""
    events = {}
    in_section = False
    for line in obs_text.split("\n"):
        if "=== ACTIVE EVENTS ===" in line:
            in_section = True
            continue
        if in_section and line.startswith("==="):
            break
        if in_section and "|" in line and "EVT-" in line:
            parts = [p.strip() for p in line.split("|")]
            if len(parts) >= 2:
                events[parts[0]] = parts[1]
    return events


TYPE_TO_VEHICLE = {"fire": "fire", "medical": "ambulance", "crime": "police", "accident": "ambulance"}

SEV_KEYWORDS = {
    5: ["not breathing", "collapsed and", "active shooter", "trapped", "mass incident",
        "massive fire", "whole block", "not moving", "pileup", "send everything"],
    4: ["won't wake", "gunshots", "flipped", "blood everywhere", "people yelling",
        "building's on fire", "kids are upstairs", "not responding"],
    3: ["chest pain", "fight", "mugged", "knife", "crash", "bleeding",
        "fire at", "flames", "cyclist", "arm feels numb"],
    2: ["fainted", "break-in", "dumpster", "fender", "small fire", "ankle",
        "shoplifter", "pale"],
}


def heuristic_severity(text: str) -> int:
    t = text.lower()
    for sev in [5, 4, 3, 2]:
        if any(kw in t for kw in SEV_KEYWORDS[sev]):
            return sev
    return 1


def heuristic_vehicle_type(text: str) -> str:
    t = text.lower()
    if any(w in t for w in ["fire", "flames", "smoke", "burning", "gas leak"]):
        return "fire"
    if any(w in t for w in ["shooter", "gunshot", "mugged", "knife", "break-in",
                             "fight", "shoplifter", "robbery"]):
        return "police"
    return "ambulance"


def pick_free_vehicle(free_vehicles: dict, vtype: str) -> str | None:
    for vid, vt in free_vehicles.items():
        if vt == vtype:
            return vid
    return None


def pick_busy_vehicle(all_vehicles: list[dict], vtype: str) -> str | None:
    """Pick the first DISPATCHED vehicle of the given type (for hold actions)."""
    for v in all_vehicles:
        if v["type"] == vtype and v["status"] == "DISPATCHED":
            return v["id"]
    # Fallback: any non-FREE vehicle of the type
    for v in all_vehicles:
        if v["type"] == vtype and v["status"] != "FREE":
            return v["id"]
    return None

### 3.2 · Generate SFT Dataset

In [ ]:
def build_ideal_action_json(gt: dict, obs_text: str) -> str:
    """Build the ideal JSON action string from ground truth + observation."""
    is_dup = gt.get("is_duplicate", False)
    severity = gt.get("severity", 1)
    vtype = gt.get("required_vehicle_type", "ambulance")

    if is_dup:
        # Find a plausible matching event from active events
        active = parse_active_events(obs_text)
        etype = gt.get("emergency_type", "")
        dup_eid = None
        for eid, et in active.items():
            if et.strip() == etype or TYPE_TO_VEHICLE.get(et.strip()) == vtype:
                dup_eid = eid
                break
        if dup_eid is None and active:
            dup_eid = list(active.keys())[0]
        return json.dumps({
            "action_type": "duplicate",
            "severity_pred": severity,
            "is_duplicate": True,
            "duplicate_of_event_id": dup_eid,
            "vehicle_type": None,
            "vehicle_id": None,
            "reroute": None,
        })

    free = parse_free_vehicles(obs_text)
    vid = pick_free_vehicle(free, vtype)

    if vid is not None:
        # Normal dispatch — free vehicle available
        return json.dumps({
            "action_type": "dispatch",
            "severity_pred": severity,
            "is_duplicate": False,
            "duplicate_of_event_id": None,
            "vehicle_type": vtype,
            "vehicle_id": vid,
            "reroute": None,
        })

    # No free vehicle of correct type → hold action
    all_vehicles = parse_all_vehicles(obs_text)
    busy_vid = pick_busy_vehicle(all_vehicles, vtype)
    if busy_vid is not None:
        return json.dumps({
            "action_type": "hold",
            "severity_pred": severity,
            "is_duplicate": False,
            "duplicate_of_event_id": None,
            "vehicle_type": vtype,
            "vehicle_id": busy_vid,
            "reroute": None,
        })

    # Last resort: dispatch with whatever is available
    fallback_vid = next(iter(free), f"{vtype}_0")
    return json.dumps({
        "action_type": "dispatch",
        "severity_pred": severity,
        "is_duplicate": False,
        "duplicate_of_event_id": None,
        "vehicle_type": vtype,
        "vehicle_id": fallback_vid,
        "reroute": None,
    })


def generate_sft_data(env, num_episodes: int = 60) -> list[dict]:
    """Run episodes with a heuristic agent, collect (obs, ideal_action) pairs."""
    examples = []
    task_ids = [1, 2, 3]

    for ep in range(num_episodes):
        task_id = task_ids[ep % len(task_ids)]
        result = env.reset(task_id=task_id)
        prev_obs_text = result.observation.prompt

        while not result.done:
            # Heuristic action to advance the environment realistically
            free = parse_free_vehicles(prev_obs_text)
            vtype = heuristic_vehicle_type(prev_obs_text)
            vid = pick_free_vehicle(free, vtype)
            sev = heuristic_severity(prev_obs_text)

            action = SmartEmergencyAction(
                action_type="dispatch",
                severity_pred=sev,
                is_duplicate=False,
                vehicle_type=vtype,
                vehicle_id=vid,
            )

            result = env.step(action)
            gt = getattr(result.observation, "metadata", {}).get("ground_truth")
            if gt is None:
                gt = result.observation.reward_breakdown  # fallback
                prev_obs_text = result.observation.prompt
                continue

            ideal_json = build_ideal_action_json(gt, prev_obs_text)

            examples.append({
                "messages": [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": prev_obs_text},
                    {"role": "assistant", "content": ideal_json},
                ]
            })

            prev_obs_text = result.observation.prompt

        if (ep + 1) % 10 == 0:
            print(f"  Episodes: {ep+1}/{num_episodes}  |  examples so far: {len(examples)}")

    return examples


print("Generating SFT data …")
sft_examples = generate_sft_data(env, num_episodes=60)
print(f"✅ Collected {len(sft_examples)} SFT examples")

In [ ]:
from datasets import Dataset

sft_dataset = Dataset.from_list(sft_examples)
print(sft_dataset)

### 3.3 · SFT Training

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
import torch

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

sft_config = SFTConfig(
    output_dir=SFT_OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    logging_steps=5,
    save_steps=50,
    max_seq_length=2048,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    bf16=True,
    report_to="none",
    dataset_text_field=None,  # using messages format
)

sft_trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=sft_dataset,
    args=sft_config,
    peft_config=lora_config,
)

In [ ]:
print("🏋️ Starting SFT training …")
sft_trainer.train()
print("✅ SFT training complete")

In [ ]:
# Merge LoRA weights and save
sft_trainer.save_model(SFT_OUTPUT_DIR)
tokenizer.save_pretrained(SFT_OUTPUT_DIR)
print(f"✅ SFT model saved to {SFT_OUTPUT_DIR}/")

# Free SFT memory before GRPO
del model, sft_trainer
torch.cuda.empty_cache()

---
# Phase 2 — GRPO (Group Relative Policy Optimization)

Now we use the SFT model as a warm start and train with RL against the live environment.

### 4.1 · Action Parsing from LLM Output

In [ ]:
def parse_llm_action(text: str) -> SmartEmergencyAction | None:
    """Try to extract a SmartEmergencyAction from raw LLM output."""
    from smart_emergency import RerouteAction

    # Try to find JSON block
    json_match = re.search(r"```json\s*(.*?)```", text, re.DOTALL)
    if json_match:
        text = json_match.group(1)
    else:
        # Try raw JSON object
        json_match = re.search(r"\{.*\}", text, re.DOTALL)
        if json_match:
            text = json_match.group(0)

    try:
        d = json.loads(text)
    except json.JSONDecodeError:
        return None

    try:
        # Parse optional reroute sub-action
        reroute = None
        if d.get("reroute") and isinstance(d["reroute"], dict):
            rd = d["reroute"]
            reroute = RerouteAction(
                vehicle_to_reroute=rd["vehicle_to_reroute"],
                from_event_id=rd["from_event_id"],
                replacement_vehicle_id=rd.get("replacement_vehicle_id"),
            )

        return SmartEmergencyAction(
            action_type=d.get("action_type", "dispatch"),
            severity_pred=int(d.get("severity_pred", 1)),
            is_duplicate=bool(d.get("is_duplicate", False)),
            duplicate_of_event_id=d.get("duplicate_of_event_id"),
            vehicle_type=d.get("vehicle_type"),
            vehicle_id=d.get("vehicle_id"),
            reroute=reroute,
        )
    except Exception:
        return None


# Fallback action when parsing fails
def fallback_action(obs_text: str) -> SmartEmergencyAction:
    free = parse_free_vehicles(obs_text)
    vtype = heuristic_vehicle_type(obs_text)
    vid = pick_free_vehicle(free, vtype)

    if vid is not None:
        return SmartEmergencyAction(
            action_type="dispatch",
            severity_pred=heuristic_severity(obs_text),
            is_duplicate=False,
            vehicle_type=vtype,
            vehicle_id=vid,
        )

    # No free vehicle → hold behind a busy one
    all_vehicles = parse_all_vehicles(obs_text)
    busy_vid = pick_busy_vehicle(all_vehicles, vtype)
    return SmartEmergencyAction(
        action_type="hold" if busy_vid else "dispatch",
        severity_pred=heuristic_severity(obs_text),
        is_duplicate=False,
        vehicle_type=vtype,
        vehicle_id=busy_vid or f"{vtype}_0",
    )

### 4.2 · Rollout Functions

In [ ]:
from trl.experimental.openenv import generate_rollout_completions

tokenizer_grpo = AutoTokenizer.from_pretrained(SFT_OUTPUT_DIR)
tokenizer_grpo.pad_token = tokenizer_grpo.eos_token


def make_user_prompt(obs_text: str) -> str:
    """Wrap the observation into the user turn."""
    return (
        f"You are the dispatcher. Read the situation below and respond with a single JSON action.\n\n"
        f"{obs_text}\n\n"
        f"Respond ONLY with a JSON object."
    )


def rollout_once(trainer, env, tokenizer, dataset_prompt, system_prompt, max_turns):
    """Execute one full episode with the model."""
    result = env.reset()
    observation = result.observation

    prompt_ids = []
    completion_ids = []
    logprobs = []
    rewards_severity = []
    rewards_duplicate = []
    rewards_vehicle_type = []
    rewards_vehicle_choice = []
    rewards_reroute = []
    rewards_format = []

    for _turn in range(max_turns):
        if result.done:
            break

        obs_text = observation.prompt or dataset_prompt
        user_prompt = make_user_prompt(obs_text)
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ]
        prompt_text = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=False,
            enable_thinking=False,
        )

        rollout_outputs = generate_rollout_completions(trainer, [prompt_text])[0]
        prompt_ids.extend(rollout_outputs["prompt_ids"])
        completion_ids.extend(rollout_outputs["completion_ids"])
        logprobs.extend(rollout_outputs["logprobs"])

        completion_text = rollout_outputs.get("text") or tokenizer.decode(
            rollout_outputs["completion_ids"], skip_special_tokens=True
        )

        # Parse action
        action = parse_llm_action(completion_text)
        parse_ok = action is not None
        if action is None:
            action = fallback_action(obs_text)

        result = env.step(action)
        observation = result.observation
        bd = observation.reward_breakdown if hasattr(observation, "reward_breakdown") else {}

        rewards_severity.append(bd.get("severity", 0.0))
        rewards_duplicate.append(bd.get("duplicate", 0.0))
        rewards_vehicle_type.append(bd.get("vehicle_type", 0.0))
        rewards_vehicle_choice.append(bd.get("vehicle_choice", 0.0))
        rewards_reroute.append(bd.get("reroute", 0.0))
        rewards_format.append(1.0 if parse_ok else -2.0)

    # Use last-turn rewards as the episode reward signal
    return {
        "prompt_ids": prompt_ids,
        "completion_ids": completion_ids,
        "logprobs": logprobs,
        "severity_reward": rewards_severity[-1] if rewards_severity else 0.0,
        "duplicate_reward": rewards_duplicate[-1] if rewards_duplicate else 0.0,
        "vehicle_type_reward": rewards_vehicle_type[-1] if rewards_vehicle_type else 0.0,
        "vehicle_choice_reward": rewards_vehicle_choice[-1] if rewards_vehicle_choice else 0.0,
        "reroute_reward": rewards_reroute[-1] if rewards_reroute else 0.0,
        "format_reward": rewards_format[-1] if rewards_format else 0.0,
    }


def rollout_func(prompts, trainer=None):
    """GRPO rollout function — called automatically by GRPOTrainer."""
    ep_prompt_ids, ep_completion_ids, ep_logprobs = [], [], []
    sev_rewards, dup_rewards, vt_rewards, vc_rewards, rr_rewards, fmt_rewards = [], [], [], [], [], []

    for prompt_text in prompts:
        episode = rollout_once(
            trainer=trainer,
            env=env,
            tokenizer=tokenizer_grpo,
            dataset_prompt=prompt_text,
            system_prompt=SYSTEM_PROMPT,
            max_turns=15,
        )
        ep_prompt_ids.append(episode["prompt_ids"])
        ep_completion_ids.append(episode["completion_ids"])
        ep_logprobs.append(episode["logprobs"])
        sev_rewards.append(episode["severity_reward"])
        dup_rewards.append(episode["duplicate_reward"])
        vt_rewards.append(episode["vehicle_type_reward"])
        vc_rewards.append(episode["vehicle_choice_reward"])
        rr_rewards.append(episode["reroute_reward"])
        fmt_rewards.append(episode["format_reward"])

    return {
        "prompt_ids": ep_prompt_ids,
        "completion_ids": ep_completion_ids,
        "logprobs": ep_logprobs,
        "severity_reward": sev_rewards,
        "duplicate_reward": dup_rewards,
        "vehicle_type_reward": vt_rewards,
        "vehicle_choice_reward": vc_rewards,
        "reroute_reward": rr_rewards,
        "format_reward": fmt_rewards,
    }

### 4.3 · Reward Wrapper Functions

In [ ]:
def reward_severity(completions, **kwargs):
    r = kwargs.get("severity_reward")
    return [float(x) for x in r] if r else [0.0] * len(completions)

def reward_duplicate(completions, **kwargs):
    r = kwargs.get("duplicate_reward")
    return [float(x) for x in r] if r else [0.0] * len(completions)

def reward_vehicle_type(completions, **kwargs):
    r = kwargs.get("vehicle_type_reward")
    return [float(x) for x in r] if r else [0.0] * len(completions)

def reward_vehicle_choice(completions, **kwargs):
    r = kwargs.get("vehicle_choice_reward")
    return [float(x) for x in r] if r else [0.0] * len(completions)

def reward_reroute(completions, **kwargs):
    r = kwargs.get("reroute_reward")
    return [float(x) for x in r] if r else [0.0] * len(completions)

def reward_format(completions, **kwargs):
    r = kwargs.get("format_reward")
    return [float(x) for x in r] if r else [0.0] * len(completions)

### 4.4 · GRPO Dataset & Config

In [ ]:
from datasets import Dataset as HFDataset
from trl import GRPOConfig, GRPOTrainer

grpo_dataset = HFDataset.from_dict({
    "prompt": ["Dispatch emergency services for incoming 911 calls."] * 500
})

grpo_config = GRPOConfig(
    num_train_epochs=1,
    learning_rate=5e-6,
    gradient_accumulation_steps=32,
    per_device_train_batch_size=1,
    warmup_steps=10,
    num_generations=2,
    max_completion_length=128,
    max_prompt_length=3000,
    use_vllm=True,
    vllm_mode="colocate",
    vllm_gpu_memory_utilization=0.3,
    output_dir=GRPO_OUTPUT_DIR,
    report_to="trackio",
    trackio_space_id=GRPO_OUTPUT_DIR,
    logging_steps=1,
    save_steps=10,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    push_to_hub=True,
)

### 4.5 · Train GRPO

In [ ]:
import torch

grpo_trainer = GRPOTrainer(
    model=SFT_OUTPUT_DIR,  # start from the SFT checkpoint
    processing_class=tokenizer_grpo,
    reward_funcs=[
        reward_severity,
        reward_duplicate,
        reward_vehicle_type,
        reward_vehicle_choice,
        reward_reroute,
        reward_format,
    ],
    train_dataset=grpo_dataset,
    args=grpo_config,
    rollout_func=rollout_func,
)

# Memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_mem = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
print(f"GPU = {gpu_stats.name}  |  Max memory = {round(gpu_stats.total_memory/1024**3, 1)} GB")
print(f"Reserved before training: {start_mem} GB")

In [ ]:
print("🏋️ Starting GRPO training …")
trainer_stats = grpo_trainer.train()
print("✅ GRPO training complete")

In [ ]:
# Memory report
used = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
print(f"Peak reserved memory: {used} GB")
print(f"Training time: {round(trainer_stats.metrics['train_runtime']/60, 1)} min")

# Save & push
env.close()
grpo_trainer.save_model(GRPO_OUTPUT_DIR)
grpo_trainer.push_to_hub()
print(f"✅ Model saved and pushed to Hub")

---
# Phase 3 — Inference & Evaluation

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer as AT

inf_model = AutoModelForCausalLM.from_pretrained(GRPO_OUTPUT_DIR, device_map="auto", torch_dtype="auto")
inf_tokenizer = AT.from_pretrained(GRPO_OUTPUT_DIR)

def run_episode(env, model, tokenizer, task_id=1):
    result = env.reset(task_id=task_id)
    obs = result.observation
    total_reward = 0.0
    step = 0

    while not result.done and step < 20:
        user_prompt = make_user_prompt(obs.prompt)
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ]
        prompt_text = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, tokenize=False, enable_thinking=False,
        )
        inputs = tokenizer([prompt_text], return_tensors="pt").to(model.device)
        gen_ids = model.generate(**inputs, max_new_tokens=256)
        output = tokenizer.decode(gen_ids[0][len(inputs.input_ids[0]):], skip_special_tokens=True)

        action = parse_llm_action(output)
        if action is None:
            action = fallback_action(obs.prompt)
            print(f"  Step {step}: ⚠️  parse fail → fallback")
        else:
            print(f"  Step {step}: {action.action_type} sev={action.severity_pred} "
                  f"vtype={action.vehicle_type} vid={action.vehicle_id}")

        result = env.step(action)
        obs = result.observation
        r = obs.reward_breakdown.get("total", 0.0) if hasattr(obs, "reward_breakdown") else 0.0
        total_reward += r
        step += 1

    print(f"\n  Episode done — total reward: {total_reward:.2f} over {step} steps")
    return total_reward

In [ ]:
print("=" * 50)
print("Running evaluation episode (Task 1 — Easy)")
print("=" * 50)
eval_env = SmartEmergencyEnv(base_url=ENV_URL).sync()
run_episode(eval_env, inf_model, inf_tokenizer, task_id=1)
eval_env.close()